# Paleoclimate model comparison — data exploration

This notebook inspects four paleoclimate temperature datasets (Beyer2020,
PalMod2, TraCE-21k, CHELSA-TraCE21k) and produces three things:

1. a per-dataset text report (`dataset_info.txt`) covering attrs, dims, coords,
   time convention, the temperature variable, and the spatial grid;
2. a 2-panel visual summary of temporal coverage and the grid cell each dataset
   returns near a requested point;
3. a comparison plot of air-temperature time series at one location.

The trickiest part is the **time axis** — every file uses a different
convention. The `time_to_ka_bp` helper centralises that translation.

## Imports

All third-party imports live here so each function cell stays focused.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Coord-name candidates used by every function that opens a NetCDF.
# Centralised here so it's edited in one place if a new dataset uses
# something exotic (e.g. "rlat"/"rlon" for rotated grids).
LAT_NAMES = ["lat", "latitude", "nav_lat", "y"]
LON_NAMES = ["lon", "longitude", "nav_lon", "x"]

## Data file paths

Paths are resolved relative to the current working directory so the notebook
works on any machine that mirrors the `data/` layout.

In [ ]:
DATA = Path.cwd() / "data"

beyer2020_air = str(DATA / "Beyer2020" / "data" / "LateQuaternary_Environment.nc")
palmod2_air   = str(DATA / "PalMod2" / "output" / "PMMXMCRTDGr111Amtasgn30201_1-250.nc")
palmod2_soil  = str(DATA / "PalMod2" / "output" / "PMMXMCRTDGr111Lmtslgn30201_1-250.nc")
trace_air     = str(DATA / "TraCE-21k" / "data" / "trace.01-36.22000BP.clm2.TSA.22000BP_decavg_400BCE.nc")
trace_soil    = str(DATA / "TraCE-21k" / "data" / "trace.01-36.22000BP.clm2.TSOI.22000BP_decavg_400BCE.nc")
chelsa_air    = str(DATA / "CHELSA-TraCE21k-centennial" / "output" / "tasmean.nc")

## Dataset registries

Two DataFrames — one for near-surface air temperature, one for soil
temperature — that the rest of the notebook iterates over. Each row is:

- `name` — short label for plots and reports
- `path` — absolute path to the NetCDF file
- `variable` — name of the temperature variable inside the file
- `unit` — `"degC"` or `"K"` (drives the K→°C conversion downstream)

In [ ]:
df_air = pd.DataFrame(
    [
        ["Beyer2020",                  beyer2020_air, "temperature", "degC"],
        ["PalMod2",                    palmod2_air,   "tas",         "K"],
        ["TraCE-21k",                  trace_air,     "TSA",         "K"],
        ["CHELSA-TraCE21k-centennial", chelsa_air,    "tasmean",     "K"],
    ],
    columns=["name", "path", "variable", "unit"],
)
df_air

In [ ]:
df_soil = pd.DataFrame(
    [
        ["PalMod2",   palmod2_soil, "tsl",  "K"],
        ["TraCE-21k", trace_soil,   "TSOI", "K"],
    ],
    columns=["name", "path", "variable", "unit"],
)
df_soil

## Time conversion: NetCDF time coord → ka BP

Each dataset stores time differently. The job of this function is to return
**ka BP with past = positive, present = 0**, so downstream code can treat all
four uniformly.

Observed conventions:

| Dataset    | `units` attribute              | raw range          | conversion to ka BP        |
|------------|--------------------------------|--------------------|----------------------------|
| Beyer2020  | `years since present`          | `[-120000, 0]`     | `-vals / 1000`             |
| PalMod2    | `days since 1-1-1 00:00:00`    | `[181, 9 130 878]` | `(max - vals) / 365250`    |
| TraCE-21k  | `ka BP` (but values negative!) | `[-22, 0.03]`      | `-vals`                    |
| CHELSA     | `days since -20010-07-01 ...`  | `[0, 8 035 335]`   | `(max - vals) / 365250`    |

For the `days since <calendar date>` files, the absolute reference date is
irrelevant — we only need "how long before present". Anchoring on
`vals.max()` (the most recent sample) and converting days → kyr is accurate
to ~0.5 ka, which is invisible on a symlog axis spanning 0–125 ka.

In [ ]:
def time_to_ka_bp(time_da):
    """Convert NetCDF time coord to ka BP (past = positive, present = 0)."""
    units = (time_da.attrs.get("units") or "").lower()
    vals = np.asarray(time_da.values, dtype=float)

    # TraCE: "ka BP" but values are negative kiloyears
    if "ka" in units or "kyr" in units:
        return -vals if vals.min() < 0 else vals

    # Beyer: "years since present" — past is negative
    if "since present" in units:
        return -vals / 1000

    # PalMod2, CHELSA: "days since <calendar date>" — anchor on last sample
    if "since" in units and "day" in units:
        return (vals.max() - vals) / 365250.0

    # Fallback: sign of the data tells the convention
    return (-vals / 1000) if vals.min() < 0 else (vals / 1000)

## Dataset inspection

`inspect_dataset` returns a multi-line string describing one NetCDF file:
global attrs, dims, coordinates, time convention (raw + converted to ka BP via
`time_to_ka_bp`), the temperature variable's metadata, and the spatial grid.

In [ ]:
def inspect_dataset(name, path, variable, unit_in_df):
    """Return a multi-line description of one NetCDF dataset."""
    lines = [
        "=" * 78,
        f"DATASET: {name}",
        "=" * 78,
        f"path     : {path}",
        f"variable : {variable}   (declared unit in df_air: {unit_in_df})",
        "",
    ]

    with xr.open_dataset(path, decode_times=False) as ds:

        # ---- global attributes ----
        lines.append("-- global attributes --")
        if ds.attrs:
            for k, v in ds.attrs.items():
                lines.append(f"  {k}: {v}")
        else:
            lines.append("  (none)")
        lines.append("")

        # ---- dimensions ----
        # Using ds.sizes (not ds.dims) avoids a FutureWarning in xarray ≥2024.
        lines.append("-- dimensions --")
        for d, n in ds.sizes.items():
            lines.append(f"  {d}: {n}")
        lines.append("")

        # ---- coordinates ----
        lines.append("-- coordinates --")
        for cname in ds.coords:
            c = ds[cname]
            vals = np.asarray(c.values)
            try:
                rng = f"[{float(vals.min()):.4f}, {float(vals.max()):.4f}]"
            except (TypeError, ValueError):
                # non-numeric coord (e.g. cftime) — fall back to endpoints
                rng = f"first={vals.flat[0]}, last={vals.flat[-1]}"
            lines.append(f"  {cname}  shape={c.shape}  dtype={c.dtype}  range={rng}")
            for ak, av in c.attrs.items():
                lines.append(f"      {ak}: {av}")
        lines.append("")

        # ---- time convention (raw + converted) ----
        # The raw range alone isn't enough — we also report what time_to_ka_bp
        # produces so the report and the plots can't drift apart.
        if "time" in ds.coords:
            t = ds["time"]
            tvals = np.asarray(t.values, dtype=float)
            ka = time_to_ka_bp(t)
            step = np.diff(tvals)
            step_info = (
                f"median = {np.median(step):.3f}, "
                f"min = {step.min():.3f}, max = {step.max():.3f}"
                if step.size else "n/a"
            )
            lines += [
                "-- time convention --",
                f"  units              : {t.attrs.get('units', '<missing>')}",
                f"  raw range          : [{tvals.min():.3f}, {tvals.max():.3f}]  (n={tvals.size})",
                f"  raw step           : {step_info}",
                f"  converted ka BP    : [{float(ka.min()):.3f}, {float(ka.max()):.3f}]",
                "",
            ]

        # ---- temperature variable ----
        lines.append(f"-- variable: {variable} --")
        if variable in ds.variables:
            v = ds[variable]
            lines.append(f"  dims    : {v.dims}")
            lines.append(f"  shape   : {v.shape}")
            lines.append(f"  dtype   : {v.dtype}")
            for ak, av in v.attrs.items():
                lines.append(f"  attr {ak}: {av}")
        else:
            lines.append(f"  !! variable {variable!r} NOT FOUND in file.")
            lines.append(f"  available data vars: {list(ds.data_vars)}")
        lines.append("")

        # ---- spatial grid (lat/lon detection mirrors the extraction logic) ----
        lat_name = next((n for n in LAT_NAMES if n in ds.coords), None)
        lon_name = next((n for n in LON_NAMES if n in ds.coords), None)
        lines.append("-- spatial grid --")
        lines.append(f"  detected lat coord: {lat_name}")
        lines.append(f"  detected lon coord: {lon_name}")
        if lat_name and lon_name:
            la = ds[lat_name].values
            lo = ds[lon_name].values
            lon_conv = "0–360" if float(lo.max()) > 180 else "-180–180"
            lat_res = float(np.median(np.abs(np.diff(la))))
            lon_res = float(np.median(np.abs(np.diff(lo))))
            lines.append(
                f"  lat range : [{float(la.min()):.3f}, {float(la.max()):.3f}]"
                f"  resolution ≈ {lat_res:.4f}°"
            )
            lines.append(
                f"  lon range : [{float(lo.min()):.3f}, {float(lo.max()):.3f}]"
                f"  resolution ≈ {lon_res:.4f}°  (convention: {lon_conv})"
            )
        lines.append("")

    return "\n".join(lines)

### Run inspection and save report

Writes one `=`-separated block per dataset to `dataset_info.txt` and prints
the same content to the cell output.

In [ ]:
out_path = Path("dataset_info.txt")
report = [inspect_dataset(r["name"], r["path"], r["variable"], r["unit"])
          for _, r in df_air.iterrows()]

out_path.write_text("\n".join(report), encoding="utf-8")
#print("\n".join(report))
#print(f"\n→ Saved full report to: {out_path.resolve()}")

## Visual summary

Two panels side by side:

- **Left** — temporal coverage on a symlog x-axis. Each bar is one dataset.
  The annotation shows median step `Δt` and sample count `n`.
- **Right** — regional map (PlateCarree, ±15° around the requested point).
  Each coloured rectangle is the grid cell that dataset returned, drawn at
  its true size in degrees. The red star marks the requested point.

Both panels share the same `collect_summary` data extraction, so they and the
text report cannot disagree.

In [ ]:
# Each dataset is keyed by name so it keeps the same colour
# whether it's plotted from df_air, df_soil, or anything else.
DATASET_COLORS = {
    "Beyer2020":                  "steelblue",
    "PalMod2":                    "darkorange",
    "TraCE-21k":                  "seagreen",
    "CHELSA-TraCE21k-centennial": "indianred",
}
DEFAULT_COLOR = "mediumpurple"   # for any dataset not in the dict

In [ ]:
def collect_summary(df, lat_req, lon_req):
    """Return a list of dicts with the minimal info needed for the summary plot."""
    rows = []
    for _, r in df.iterrows():
        with xr.open_dataset(r["path"], decode_times=False) as ds:
            # Time — same conversion as the comparison plot uses.
            ka = np.sort(time_to_ka_bp(ds["time"]))
            step_kyr = float(np.median(np.diff(ka))) if ka.size > 1 else np.nan

            # Spatial — figure out which cell .sel(method="nearest") would pick.
            lat_name = next(n for n in LAT_NAMES if n in ds.coords)
            lon_name = next(n for n in LON_NAMES if n in ds.coords)

            # Handle 0–360 vs -180–180 longitude conventions.
            lon_q = lon_req + 360 if float(ds[lon_name].max()) > 180 and lon_req < 0 else lon_req

            sub = ds[r["variable"]].sel(
                {lat_name: lat_req, lon_name: lon_q}, method="nearest"
            )
            cell_lat = float(sub[lat_name])
            cell_lon = float(sub[lon_name])
            if cell_lon > 180:                    # show in -180..180 for plotting
                cell_lon -= 360

            lat_res = float(np.median(np.abs(np.diff(ds[lat_name].values))))
            lon_res = float(np.median(np.abs(np.diff(ds[lon_name].values))))

        rows.append({
            "name": r["name"],
            "t_min": float(ka.min()), "t_max": float(ka.max()),
            "n_steps": int(ka.size), "step_kyr": step_kyr,
            "cell_lat": cell_lat, "cell_lon": cell_lon,
            "lat_res": lat_res, "lon_res": lon_res,
        })
    return rows

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

def plot_dataset_summary(df, lat_req, lon_req):
    rows = collect_summary(df, lat_req, lon_req)

    # Side-by-side: temporal coverage left, regional map right.
    # constrained_layout handles cartopy axes correctly (tight_layout doesn't).
    fig = plt.figure(figsize=(15, max(5, 0.55 * len(rows) + 3)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.4, 1])
    ax_t = fig.add_subplot(gs[0, 0])
    ax_m = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())

    # ---- Panel 1: temporal coverage ----
    for i, r in enumerate(rows):
        c = DATASET_COLORS.get(r["name"], DEFAULT_COLOR)
        ax_t.hlines(i, r["t_min"], r["t_max"], colors=c, lw=7, alpha=0.85)
        ax_t.text(
            r["t_max"], i + 0.25,
            f"  Δt ≈ {r['step_kyr']*1000:.0f} yr   n={r['n_steps']}",
            va="bottom", ha="left", fontsize=9, color="0.4",
        )
    ax_t.set_yticks(range(len(rows)))
    ax_t.set_yticklabels([r["name"] for r in rows])
    #ax_t.set_xscale("symlog", linthresh=0.1)
    #ax_t.set_xticks([0, 0.1, 1, 10, 100])
    #ax_t.set_xticklabels(["0", "0.1", "1", "10", "100"])
    ax_t.set_xlim(150, -0.1)
    ax_t.invert_yaxis()
    ax_t.set_xlabel("Time (ka BP)")
    ax_t.set_title("Temporal coverage")
    ax_t.grid(alpha=0.25, which="both", axis="x", linewidth=0.5)
    ax_t.spines["top"].set_visible(False)
    ax_t.spines["right"].set_visible(False)

    # ---- Panel 2: regional map ----
    pc = ccrs.PlateCarree()
    MARGIN = 5
    ax_m.set_extent(
        [lon_req - MARGIN, lon_req + MARGIN,
         lat_req - MARGIN, lat_req + MARGIN],
        crs=pc,
    )
    ax_m.add_feature(cfeature.LAND,      facecolor="#EAE6DE")
    ax_m.add_feature(cfeature.OCEAN,     facecolor="#D6E4F0")
    ax_m.add_feature(cfeature.COASTLINE, lw=0.4, edgecolor="0.4")
    ax_m.add_feature(cfeature.BORDERS,   lw=0.3, edgecolor="0.5", alpha=0.6)
    ax_m.gridlines(lw=0.3, color="0.7", alpha=0.5, draw_labels=True)

    for r in rows:
        c = DATASET_COLORS.get(r["name"], DEFAULT_COLOR)
        ax_m.add_patch(Rectangle(
            (r["cell_lon"] - r["lon_res"] / 2, r["cell_lat"] - r["lat_res"] / 2),
            r["lon_res"], r["lat_res"],
            edgecolor=c, facecolor=c, alpha=0.35, lw=1.4,
            transform=pc, zorder=3,
            label=f"{r['name']}  ({r['lat_res']:.2f}°×{r['lon_res']:.2f}°)",
        ))
        ax_m.scatter([r["cell_lon"]], [r["cell_lat"]],
                     color=c, s=35, zorder=4, alpha=0.9, transform=pc)

    ax_m.scatter([lon_req], [lat_req], marker="*", s=260,
                 color="crimson", edgecolor="k", lw=0.6,
                 transform=pc, zorder=5, label="requested")

    ax_m.set_title(f"Grid cells near ({lat_req:.2f}°N, {lon_req:.2f}°E)")
    ax_m.legend(fontsize=8, loc="lower left", frameon=False)

    # No plt.tight_layout() — constrained_layout=True handles it.
    plt.show()
    return rows

In [ ]:
#summary = plot_dataset_summary(df_air, lat_req=52.52, lon_req=13.40)  # Berlin

## Temperature extraction at a point

`extract_temperature_at_point` opens one NetCDF, picks the grid cell nearest
to a requested lat/lon, collapses the seasonal cycle if a `month` dimension
exists, converts K → °C if needed, and returns the resulting 1-D series in
ka BP coordinates.

In [ ]:
DEPTH_NAMES = ["levsoi", "depth"]   # the two your files use

def extract_temperature_at_point(path, temp_var, lat, lon, input_unit="degC"):
    """Return (time_ka, y_degC, cell_lat, cell_lon) for one NetCDF file.

    Always picks the topmost soil layer if a depth dim is present.
    """
    with xr.open_dataset(path, decode_times=False) as ds:
        lat_name = next(n for n in LAT_NAMES if n in ds.coords)
        lon_name = next(n for n in LON_NAMES if n in ds.coords)

        # handle 0–360 vs -180–180 longitude conventions
        lon_q = lon + 360 if float(ds[lon_name].max()) > 180 and lon < 0 else lon

        da = ds[temp_var].sel({lat_name: lat, lon_name: lon_q}, method="nearest")
        cell_lat = float(da[lat_name])
        cell_lon = float(da[lon_name])

        # collapse extra dims — topmost soil layer, annual mean
        for d in DEPTH_NAMES:
            if d in da.dims:
                da = da.isel({d: 0})
                break
        if "month" in da.dims:
            da = da.mean(dim="month")
        if input_unit == "K":
            da = da - 273.15

        return time_to_ka_bp(ds["time"]), da.values.copy(), cell_lat, cell_lon

## Comparison plot

Overlays the temperature time series from every row of `df_air` (or `df_soil`)
on a single axis. The symlog x-axis expands the Holocene so recent changes
are visible alongside the full glacial cycle.

In [ ]:
def plot_temperature_comparison(df, lat, lon, title=None, log_x=True):
    """Overlay temperature time series from several NetCDF files on one axis."""
    fig, ax = plt.subplots(figsize=(11, 5))
    results = {}

    for _, row in df.iterrows():
        time_ka, y, cell_lat, cell_lon = extract_temperature_at_point(
            row["path"], row["variable"], lat, lon, input_unit=row["unit"],
        )
        ax.plot(
            time_ka, y,
            color=DATASET_COLORS.get(row["name"], DEFAULT_COLOR),
            lw=1.8, alpha=0.7,                          # transparent so overlaps show
            label=f"{row['name']}  ({cell_lat:.2f}°N, {cell_lon:.2f}°E)",
        )
        results[row["name"]] = (time_ka, y)

    if log_x:
        ax.set_xscale("symlog", linthresh=0.1)
        ax.set_xticks([0, 0.1, 1, 10, 100])
        ax.set_xticklabels(["0", "0.1", "1", "10", "100"])

    ax.invert_xaxis()
    ax.axhline(0, color="0.3", lw=0.6, ls="--", alpha=0.5)
    ax.set_xlabel("Time (ka BP)")
    ax.set_ylabel("Temperature (°C)")
    ax.set_title(title or f"Air temperature at ({lat:.2f}°N, {lon:.2f}°E)")
    ax.grid(alpha=0.25, which="both", linewidth=0.5)
    ax.spines["top"].set_visible(False)                 # cleaner frame
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, loc="best")                # no boxy legend
    plt.tight_layout()
    plt.show()
    return results

In [ ]:
#results = plot_temperature_comparison(df_air, lat=52.52, lon=13.40)  # Berlin

In [ ]:
cities = {
    "Berlin":     (52.52,  13.40),
    "Paris":      (48.85,   2.35),
    "Moscow":     (55.75,  37.62),
    "Cape Town": (-33.92,  18.42),
    "New York":   (40.71, -74.01),
    "Madrid":     (40.42,  -3.70),
    "Los Angeles":(34.05,-118.24),
    "Kampala":    ( 0.32,  32.58),
    "Mumbai":     (19.08,  72.88),
    "Bangkok":    (13.76, 100.50),
    "Tokyo":      (35.68, 139.65),
    "Helsinki":   (60.17,  24.94),
    "Oslo":       (59.91,  10.75),
    "Reykjavik":  (64.15, -21.94),
}

results = {}
summaries = {}                          # new name; the old `summary` is a list
for name, (lat, lon) in cities.items():
    print(f"--- {name} ---")
    results[name]   = plot_temperature_comparison(df_air, lat=lat, lon=lon)
    summaries[name] = plot_dataset_summary(df_air, lat_req=lat, lon_req=lon)

In [ ]:
soil_results = {}
soil_summaries = {}
for name, (lat, lon) in cities.items():
    print(f"--- {name} ---")
    soil_results[name]   = plot_temperature_comparison(df_soil, lat=lat, lon=lon)
    soil_summaries[name] = plot_dataset_summary(df_soil, lat_req=lat, lon_req=lon)